## Calidad de datos y Monitoreo de Pipeline

In [12]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.alertas import ultimas_alertas_calidad
from src.calidad import ejecutar_todas_las_validaciones, resumen_validacion, validar_esquema, validar_nulos, validar_tipos, validar_rangos
from src.config import CARPETA_DATOS, resumen_configuracion
from src.contratos import CONTRATOS
from src.db import probar_conexion
from src.pipeline import (
    CARPETA_CORRUPTOS,
    ErrorCalidadDatos,
    ejecutar_pipeline,
    validar_csv,
)

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 120)

print("Raiz del proyecto:", RAIZ)
print()
print(resumen_configuracion())
print()
print(probar_conexion())
print()
print("Contratos disponibles:", list(CONTRATOS))
print("CSV oficiales :", CARPETA_DATOS)
print("CSV corruptos :", CARPETA_CORRUPTOS)

Raiz del proyecto: c:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_1\Clase_2_persistencia

PostgreSQL : postgres@localhost:5432/ventas_devsenior
MongoDB    : base 'ml_metadata' (URI configurada: si)
Datos CSV  : C:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_1\Clase_2_persistencia\data\raw

PostgreSQL 17.4 on x86_64-windows, compiled by msvc-19.42.34436, 64-bit

Contratos disponibles: ['clientes', 'productos', 'ventas']
CSV oficiales : C:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_1\Clase_2_persistencia\data\raw
CSV corruptos : C:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_1\Clase_2_persistencia\data\raw\corruptos


## Cinco dimensiones para medir la calidad

Un contrato no “limpia” el CSV: solo dice **qué esperamos**.
`src/calidad.py` recorre el archivo y responde cinco preguntas. Si alguna falla, esa incidencia entra en una lista. El pipeline no carga y deja una **alerta** en `auditoria`.

| # | Dimensión | Pregunta | Función | Ejemplo que dispara alerta |
|---|---|---|---|---|
| 1 | **Esquema** | ¿Están las columnas justas, ni más ni menos? | `validar_esquema` | Falta `email` o aparece una columna nueva (`schema drift`) |
| 2 | **Tipos** | ¿Cada columna tiene el tipo acordado? | `validar_tipos` | `precio = "abc"` (Pandas lo lee como texto) |
| 3 | **Completitud** | ¿Las columnas obligatorias vienen llenas? | `validar_nulos` | `nombre` o `fecha_registro` en blanco |
| 4 | **Dominio** | ¿El valor está en el rango o catálogo permitido? | `validar_rangos` | `descuento = 1.5`, `segmento = "diamante"`, `stock = -3` |
| 5 | **Unicidad** | ¿El identificador no se repite? | `validar_duplicados_id` | Dos filas con el mismo `cliente_id` |

### De la incidencia a la alerta

```
CSV  →  las 5 validaciones  →  lista de incidencias
                                 │
                    vacía ───────┼─────── con elementos
                       │                    │
              alerta estado = exito    alerta estado = error
              (se puede cargar)        FAIL FAST: no se toca PostgreSQL
```

La alerta no es un email: es una fila en `auditoria` (`operacion = VALIDACION`).
`ultimas_alertas_calidad()` es el monitor de esta clase: quién validó, qué tabla y qué rompió el contrato.

## El contrato de los datos

In [8]:
contrato_ventas = CONTRATOS["ventas"]

print("Tabla destino:", contrato_ventas.tabla)
print("Archivo csv:", contrato_ventas.archivo_csv)
print("Columnas que ESPERAMOS, en el orden: \n")

for col in contrato_ventas.columnas:
    extra = []
    if col.valores_permitidos:
        extra.append(f"permitidos={col.valores_permitidos}")
    
    if col.minimo is not None:
        extra.append(f"min={col.minimo}")
    
    if col.maximo is not None:
        extra.append(f"max={col.maximo}")
    
    detalle = f"  ({','.join(extra)})" if extra else ""
    
    print(f"   -{col.nombre:<18} tipo={col.tipo:<6}{detalle}")
    

Tabla destino: ventas
Archivo csv: ventas.csv
Columnas que ESPERAMOS, en el orden: 

   -venta_id           tipo=int     (min=1)
   -cliente_id         tipo=int     (min=1)
   -producto_id        tipo=int     (min=1)
   -fecha_venta        tipo=date  
   -cantidad           tipo=int     (min=1)
   -precio_unitario    tipo=float   (min=0)
   -descuento          tipo=float   (min=0,max=1)
   -total              tipo=float   (min=0)
   -canal              tipo=str     (permitidos=['web', 'app', 'tienda', 'telefono'])


In [11]:
datos_ventas, indicidencias_oficiales = validar_csv("ventas")

print(f"Filas leidas: {len(datos_ventas):,}")
print(resumen_validacion(indicidencias_oficiales))
print("\nValidación solo de Esquema (columnas)")
print(validar_esquema(datos_ventas, CONTRATOS["ventas"]) or " (sin incidiencias)")


Filas leidas: 200,000
OK: el archivo cumple el contrato.

Validación solo de Esquema (columnas)
 (sin incidiencias)


## Tipos, nulos y rangos

In [ ]:
contrato = CONTRATOS["ventas"]

print("TIPOS     :",  validar_tipos(datos_ventas, contrato) or "OK")
print("NULOS     :",  validar_nulos(datos_ventas, contrato) or "OK")
print("RANGOS    :",  validar_rangos(datos_ventas, contrato) or "OK")
print()
print("TODAS juntas:")
print(resumen_validacion(ejecutar_todas_las_validaciones(datos_ventas, contrato)))

print("\nResumen por tabla oficial:")
for tabla in ("clientes", "productos", "ventas"):
    _, incidencias = validar_csv(tabla)
    print(f" {tabla:<10} {resumen_validacion(incidencias)}")

TIPOS     : OK
NULOS     : OK
RANGOS    : OK

TODAS juntas:
OK: el archivo cumple el contrato.

Resumen por tabla oficial:
 clientes   OK: el archivo cumple el contrato.
 productos  OK: el archivo cumple el contrato.
 ventas     OK: el archivo cumple el contrato.


## Excepciones y fail fast

In [16]:
try:
    resultado = ejecutar_pipeline("ventas", cargar_si_ok=False)
    print("Pipeline OK")
    print("tabla        :", resultado["tabla"])
    print("filas        :", f"{resultado['filas']:,}")
    print("cargado      :", resultado["cargado"])
    print("mensaje       :", resultado["mensaje"])

except ErrorCalidadDatos as error:
    print("Esto no debería ocurrir con el CSV oficial")
    print(error)

Pipeline OK
tabla        : ventas
filas        : 200,000
cargado      : False
mensaje       : OK: el archivo cumple el contrato.


## Alertas 

Schema drift = el archivo de hoy no tiene la mimsa forma que el de ayer.

In [ ]:
#Tres arhcivos rotos. 

casos = [
    ("clientes", CARPETA_CORRUPTOS / "clientes_sin_email.csv", "Falta la columna email"),
    ("ventas", CARPETA_CORRUPTOS / "ventas_columna_extra.csv", "Columan extra (schema drift)"),
    ("ventas", CARPETA_CORRUPTOS / "ventas_precio_texto.csv", "Precio como texto"),
]

for tabla, ruta, titulo in casos:
    print('=' * 50)
    print(titulo)
    print("Archivo:", ruta.name)
    try:
        ejecutar_pipeline(tabla, ruta_alternativa=ruta, cargar_si_ok=False)
        print(" Se cargo (esto no debería pasar)")
    
    except ErrorCalidadDatos as error:
        print(" RECHAZADO por calidad:")
        for incidencia in error.incidencias:
            print("   -", incidencia)
    
    except FileNotFoundError:
        print(" Falta el csv corrupto. Ejecuta:")
        print(" Python scripts/generate_datos_corruptos.py")

Falta la columna email
Archivo: clientes_sin_email.csv
 RECHAZADO por calidad:
   - Columnas FALTANTES: ['email']
Columan extra (schema drift)
Archivo: ventas_columna_extra.csv
 RECHAZADO por calidad:
   - Columnas EXTRA (schema drift): ['promocion']
Precio como texto
Archivo: ventas_precio_texto.csv
 RECHAZADO por calidad:
   - Columna 'precio_unitario': tipo esperado float, recibido object


## Monitoreo: lectura de alertas

In [31]:
# Foto de monitoreo: que alertas de calidad quedaorn registradas

from IPython.display import display

columnas = [
    "fecha_hora",
    "proceso",
    "operacion",
    "tabla_afectada",
    "estado",
    "registros_procesados",
    "mensaje",
]

alertas = ultimas_alertas_calidad(limite=8)
if alertas.empty:
    print("Aún no hay alertas. Ejecuta las celdas del bloque")

else:
    display(alertas[columnas])
    

,fecha_hora,proceso,operacion,tabla_afectada,estado,registros_procesados,mensaje
0,2026-09-13 21:00:26.458693,pipeline_calidad,VALIDACION,ventas,error,30,ErrorCalidadDatos: Fallo de calidad: Columna '...
1,2026-09-13 21:00:26.453758,pipeline_calidad,VALIDACION,ventas,error,30,ALERTA CALIDAD: Columna 'precio_unitario': tip...
2,2026-09-13 21:00:26.427135,pipeline_calidad,VALIDACION,ventas,error,30,ErrorCalidadDatos: Fallo de calidad: Columnas ...
3,2026-09-13 21:00:26.422005,pipeline_calidad,VALIDACION,ventas,error,30,ALERTA CALIDAD: Columnas EXTRA (schema drift):...
4,2026-09-13 21:00:26.398101,pipeline_calidad,VALIDACION,clientes,error,30,ErrorCalidadDatos: Fallo de calidad: Columnas ...
5,2026-09-13 21:00:26.393904,pipeline_calidad,VALIDACION,clientes,error,30,ALERTA CALIDAD: Columnas FALTANTES: ['email']
6,2026-09-13 21:00:07.604607,pipeline_calidad,VALIDACION,ventas,error,0,FileNotFoundError: [Errno 2] No such file or d...
7,2026-09-13 21:00:07.599793,pipeline_calidad,VALIDACION,ventas,error,30,ErrorCalidadDatos: Fallo de calidad: Columnas ...


Validar antes de cargar. Alertar cuando el formato cambia. Hacer Auditoría.